In [1]:
import pandas as pd
import numpy as np

print("="*80)
print("🔧 CRÉATION FEATURES_COMPTES AVEC NOUVELLES VARIABLES")
print("="*80)

# ============================================================================
# ÉTAPE 1: CHARGER ET PRÉPARER DONNÉES
# ============================================================================

print("\n1. CHARGEMENT DES DONNÉES")
print("-"*80)

operations = pd.read_excel(
    r"C:\Users\Pc\Desktop\new\opérations comptes.xls",
    engine="xlrd",
    dtype={'compte': str}
)

# CRITIQUE: Préserver les comptes EXACTS (pas d'arrondi scientifique)
operations['compte'] = operations['compte'].astype(str)

print(f"✓ Opérations chargées: {operations.shape}")
print(f"✓ Comptes uniques: {operations['compte'].nunique()}")
print(f"✓ Exemple comptes:")
print(operations['compte'].head(10).tolist())

# Conversion date
operations['date_operation'] = pd.to_datetime(operations['date_operation'])
operations['mois'] = operations['date_operation'].dt.to_period('M')
operations['heure'] = operations['date_operation'].dt.hour
operations['jour_semaine'] = operations['date_operation'].dt.dayofweek
operations['weekend'] = operations['jour_semaine'].isin([5, 6])
operations['nocturne'] = (operations['heure'] >= 22) | (operations['heure'] < 6)

# Trier
operations = operations.sort_values(['compte', 'date_operation'])

print(f"✓ Données préparées")

🔧 CRÉATION FEATURES_COMPTES AVEC NOUVELLES VARIABLES

1. CHARGEMENT DES DONNÉES
--------------------------------------------------------------------------------
✓ Opérations chargées: (11309, 10)
✓ Comptes uniques: 2839
✓ Exemple comptes:
['14010123119700000768', '14010123110716486824', '14010123110726899712', '14010123110700081311', '14010123110719634183', '14010123110705600512', '14010123110741711771', '14010123110719989106', '14010123110725799936', '14010123110715637589']
✓ Données préparées


In [2]:
# ============================================================================
#FEATURES DE BASE (MONTANTS)
# ============================================================================

print("\n2. FEATURES DE BASE")
print("-"*80)

features = operations.groupby('compte').agg(
    nb_transactions=('montant', 'count'),
    total_montant=('montant', 'sum'),
    montant_moyen=('montant', 'mean'),
    montant_median=('montant', 'median'),
    montant_std=('montant', 'std'),
    montant_min=('montant', 'min'),
    montant_max=('montant', 'max'),
    date_min=('date_operation', 'min'),
    date_max=('date_operation', 'max')
).reset_index()


2. FEATURES DE BASE
--------------------------------------------------------------------------------


In [3]:
# Durée activité
features['duree_activite_jours'] = (
    features['date_max'] - features['date_min']
).dt.days + 1

# Ratio max/moyen
features['ratio_max_moyen'] = features['montant_max'] / features['montant_moyen']

print(f"✓ Features de base créées: {features.shape}")

✓ Features de base créées: (2839, 12)


In [4]:
# ============================================================================
# ÉTAPE 3: FEATURES TEMPORELLES
# ============================================================================

print("\n3. FEATURES TEMPORELLES")
print("-"*80)

# Transactions par mois
tx_par_mois = (
    operations.groupby(['compte', 'mois'])
    .size()
    .groupby('compte')
    .mean()
    .reset_index(name='nb_transactions_mois')
)

# Pourcentages nocturne/weekend
temp_features = operations.groupby('compte').agg(
    pct_nocturne=('nocturne', 'mean'),
    pct_weekend=('weekend', 'mean')
).reset_index()

# Fusionner
features = features.merge(tx_par_mois, on='compte', how='left')
features = features.merge(temp_features, on='compte', how='left')

print(f"✓ Features temporelles ajoutées")


3. FEATURES TEMPORELLES
--------------------------------------------------------------------------------
✓ Features temporelles ajoutées


In [5]:
# ============================================================================
# ÉTAPE 4: FEATURES GÉOGRAPHIQUES
# ============================================================================

print("\n4. FEATURES GÉOGRAPHIQUES")
print("-"*80)

# Nombre de lieux différents
geo_features = operations.groupby('compte').agg(
    nb_lieux_differents=('Lieu', 'nunique')
).reset_index()

# Lieu principal
lieu_principal = (
    operations.groupby(['compte', 'Lieu'])
    .size()
    .reset_index(name='count')
    .sort_values(['compte', 'count'], ascending=[True, False])
    .drop_duplicates('compte')
    [['compte', 'Lieu', 'count']]
    .rename(columns={'Lieu': 'lieu_principal'})
)

# Pourcentage lieu principal
total_ops = operations.groupby('compte').size().reset_index(name='total_ops')
lieu_principal = lieu_principal.merge(total_ops, on='compte')
lieu_principal['pct_lieu_principal'] = (
    lieu_principal['count'] / lieu_principal['total_ops']
)
lieu_principal = lieu_principal.drop(columns=['count', 'total_ops'])

# Changements de lieu
operations['lieu_change'] = (
    operations.groupby('compte')['Lieu'].shift() != operations['Lieu']
)
mobilite = operations.groupby('compte')['lieu_change'].mean().reset_index(
    name='changements_lieu_frequents'
)

# Fusionner
features = features.merge(geo_features, on='compte', how='left')
features = features.merge(lieu_principal, on='compte', how='left')
features = features.merge(mobilite, on='compte', how='left')

print(f"✓ Features géographiques ajoutées")



4. FEATURES GÉOGRAPHIQUES
--------------------------------------------------------------------------------
✓ Features géographiques ajoutées


In [6]:
# ============================================================================
# ÉTAPE 5: FEATURES TEMPS ENTRE TRANSACTIONS
# ============================================================================

print("\n5. FEATURES TEMPS ENTRE TRANSACTIONS")
print("-"*80)

# Temps entre transactions (minutes)
operations['delta_temps_min'] = (
    operations.groupby('compte')['date_operation']
    .diff()
    .dt.total_seconds() / 60
)

features_temps = operations.groupby('compte').agg(
    temps_moyen_entre_tx_min=('delta_temps_min', 'mean'),
    temps_median_entre_tx_min=('delta_temps_min', 'median'),
    temps_min_entre_tx_min=('delta_temps_min', 'min'),
    temps_std_entre_tx_min=('delta_temps_min', 'std')
).reset_index()

# Fusionner
features = features.merge(features_temps, on='compte', how='left')

print(f"✓ Features temps ajoutées")



5. FEATURES TEMPS ENTRE TRANSACTIONS
--------------------------------------------------------------------------------
✓ Features temps ajoutées


In [7]:
# ============================================================================
# ÉTAPE 6: 🆕 NOUVELLES VARIABLES - MONTANTS CROISSANTS
# ============================================================================

print("\n6. 🆕 NOUVELLE VARIABLE: MONTANTS CROISSANTS CONSÉCUTIFS")
print("-"*80)

croissants_list = []

for compte in operations['compte'].unique():
    compte_ops = operations[operations['compte'] == compte].copy()
    compte_ops = compte_ops.sort_values('date_operation')
    
    if len(compte_ops) < 2:
        croissants_list.append({
            'compte': compte,
            'montants_croissants_consecutifs': 0,
            'ratio_montant_max_min': 1
        })
        continue
    
    # Compter séquences croissantes
    montants = compte_ops['montant'].values
    max_croissants = 0
    croissants_actuels = 0
    
    for i in range(len(montants) - 1):
        if montants[i+1] > montants[i]:
            croissants_actuels += 1
            max_croissants = max(max_croissants, croissants_actuels)
        else:
            croissants_actuels = 0
    
    # Ratio max/min
    ratio = montants.max() / montants.min() if montants.min() > 0 else 1
    
    croissants_list.append({
        'compte': compte,
        'montants_croissants_consecutifs': max_croissants,
        'ratio_montant_max_min': ratio
    })

df_croissants = pd.DataFrame(croissants_list)
features = features.merge(df_croissants, on='compte', how='left')

print(f"✓ Montants croissants ajoutés")


6. 🆕 NOUVELLE VARIABLE: MONTANTS CROISSANTS CONSÉCUTIFS
--------------------------------------------------------------------------------
✓ Montants croissants ajoutés


In [8]:
# ============================================================================
# ÉTAPE 7:  - VÉLOCITÉ MAXIMALE
# ============================================================================

print("\n7.  NOUVELLE VARIABLE: VÉLOCITÉ MAXIMALE (km/h)")
print("-"*80)

# Distances entre villes tunisiennes (km)
# Distances complètes entre les 24 gouvernorats tunisiens (en km)
distances = {
    # Depuis Tunis
    ('Tunis', 'Ariana'): 15,
    ('Tunis', 'Ben Arous'): 20,
    ('Tunis', 'Manouba'): 10,
    ('Tunis', 'Nabeul'): 65,
    ('Tunis', 'Zaghouan'): 60,
    ('Tunis', 'Bizerte'): 65,
    ('Tunis', 'Béja'): 105,
    ('Tunis', 'Jendouba'): 155,
    ('Tunis', 'Kef'): 175,
    ('Tunis', 'Siliana'): 125,
    ('Tunis', 'Sousse'): 140,
    ('Tunis', 'Monastir'): 160,
    ('Tunis', 'Mahdia'): 200,
    ('Tunis', 'Sfax'): 270,
    ('Tunis', 'Kairouan'): 160,
    ('Tunis', 'Kasserine'): 300,
    ('Tunis', 'Sidi Bouzid'): 265,
    ('Tunis', 'Gabès'): 405,
    ('Tunis', 'Médenine'): 480,
    ('Tunis', 'Tataouine'): 520,
    ('Tunis', 'Gafsa'): 340,
    ('Tunis', 'Tozeur'): 430,
    ('Tunis', 'Kébili'): 490,
    
    # Depuis Ariana
    ('Ariana', 'Ben Arous'): 25,
    ('Ariana', 'Manouba'): 12,
    ('Ariana', 'Nabeul'): 70,
    ('Ariana', 'Zaghouan'): 65,
    ('Ariana', 'Bizerte'): 60,
    ('Ariana', 'Béja'): 110,
    ('Ariana', 'Jendouba'): 160,
    ('Ariana', 'Kef'): 180,
    ('Ariana', 'Siliana'): 130,
    ('Ariana', 'Sousse'): 145,
    ('Ariana', 'Monastir'): 165,
    ('Ariana', 'Mahdia'): 205,
    ('Ariana', 'Sfax'): 275,
    ('Ariana', 'Kairouan'): 165,
    ('Ariana', 'Kasserine'): 305,
    ('Ariana', 'Sidi Bouzid'): 270,
    ('Ariana', 'Gabès'): 410,
    ('Ariana', 'Médenine'): 485,
    ('Ariana', 'Tataouine'): 525,
    ('Ariana', 'Gafsa'): 345,
    ('Ariana', 'Tozeur'): 435,
    ('Ariana', 'Kébili'): 495,
    
    # Depuis Ben Arous
    ('Ben Arous', 'Manouba'): 18,
    ('Ben Arous', 'Nabeul'): 55,
    ('Ben Arous', 'Zaghouan'): 50,
    ('Ben Arous', 'Bizerte'): 75,
    ('Ben Arous', 'Béja'): 115,
    ('Ben Arous', 'Jendouba'): 165,
    ('Ben Arous', 'Kef'): 185,
    ('Ben Arous', 'Siliana'): 135,
    ('Ben Arous', 'Sousse'): 135,
    ('Ben Arous', 'Monastir'): 155,
    ('Ben Arous', 'Mahdia'): 195,
    ('Ben Arous', 'Sfax'): 265,
    ('Ben Arous', 'Kairouan'): 155,
    ('Ben Arous', 'Kasserine'): 295,
    ('Ben Arous', 'Sidi Bouzid'): 260,
    ('Ben Arous', 'Gabès'): 400,
    ('Ben Arous', 'Médenine'): 475,
    ('Ben Arous', 'Tataouine'): 515,
    ('Ben Arous', 'Gafsa'): 335,
    ('Ben Arous', 'Tozeur'): 425,
    ('Ben Arous', 'Kébili'): 485,
    
    # Depuis Manouba
    ('Manouba', 'Nabeul'): 65,
    ('Manouba', 'Zaghouan'): 55,
    ('Manouba', 'Bizerte'): 60,
    ('Manouba', 'Béja'): 100,
    ('Manouba', 'Jendouba'): 150,
    ('Manouba', 'Kef'): 170,
    ('Manouba', 'Siliana'): 120,
    ('Manouba', 'Sousse'): 135,
    ('Manouba', 'Monastir'): 155,
    ('Manouba', 'Mahdia'): 195,
    ('Manouba', 'Sfax'): 265,
    ('Manouba', 'Kairouan'): 155,
    ('Manouba', 'Kasserine'): 295,
    ('Manouba', 'Sidi Bouzid'): 260,
    ('Manouba', 'Gabès'): 400,
    ('Manouba', 'Médenine'): 475,
    ('Manouba', 'Tataouine'): 515,
    ('Manouba', 'Gafsa'): 335,
    ('Manouba', 'Tozeur'): 425,
    ('Manouba', 'Kébili'): 485,
    
    # Depuis Nabeul
    ('Nabeul', 'Zaghouan'): 45,
    ('Nabeul', 'Bizerte'): 110,
    ('Nabeul', 'Béja'): 150,
    ('Nabeul', 'Jendouba'): 200,
    ('Nabeul', 'Kef'): 220,
    ('Nabeul', 'Siliana'): 170,
    ('Nabeul', 'Sousse'): 80,
    ('Nabeul', 'Monastir'): 100,
    ('Nabeul', 'Mahdia'): 140,
    ('Nabeul', 'Sfax'): 210,
    ('Nabeul', 'Kairouan'): 120,
    ('Nabeul', 'Kasserine'): 260,
    ('Nabeul', 'Sidi Bouzid'): 225,
    ('Nabeul', 'Gabès'): 345,
    ('Nabeul', 'Médenine'): 420,
    ('Nabeul', 'Tataouine'): 460,
    ('Nabeul', 'Gafsa'): 300,
    ('Nabeul', 'Tozeur'): 390,
    ('Nabeul', 'Kébili'): 430,
    
    # Depuis Zaghouan
    ('Zaghouan', 'Bizerte'): 105,
    ('Zaghouan', 'Béja'): 145,
    ('Zaghouan', 'Jendouba'): 195,
    ('Zaghouan', 'Kef'): 215,
    ('Zaghouan', 'Siliana'): 140,
    ('Zaghouan', 'Sousse'): 85,
    ('Zaghouan', 'Monastir'): 105,
    ('Zaghouan', 'Mahdia'): 145,
    ('Zaghouan', 'Sfax'): 215,
    ('Zaghouan', 'Kairouan'): 100,
    ('Zaghouan', 'Kasserine'): 240,
    ('Zaghouan', 'Sidi Bouzid'): 205,
    ('Zaghouan', 'Gabès'): 350,
    ('Zaghouan', 'Médenine'): 425,
    ('Zaghouan', 'Tataouine'): 465,
    ('Zaghouan', 'Gafsa'): 285,
    ('Zaghouan', 'Tozeur'): 375,
    ('Zaghouan', 'Kébili'): 435,
    
    # Depuis Bizerte
    ('Bizerte', 'Béja'): 75,
    ('Bizerte', 'Jendouba'): 100,
    ('Bizerte', 'Kef'): 140,
    ('Bizerte', 'Siliana'): 120,
    ('Bizerte', 'Sousse'): 180,
    ('Bizerte', 'Monastir'): 200,
    ('Bizerte', 'Mahdia'): 240,
    ('Bizerte', 'Sfax'): 310,
    ('Bizerte', 'Kairouan'): 200,
    ('Bizerte', 'Kasserine'): 340,
    ('Bizerte', 'Sidi Bouzid'): 305,
    ('Bizerte', 'Gabès'): 445,
    ('Bizerte', 'Médenine'): 520,
    ('Bizerte', 'Tataouine'): 560,
    ('Bizerte', 'Gafsa'): 380,
    ('Bizerte', 'Tozeur'): 470,
    ('Bizerte', 'Kébili'): 530,
    
    # Depuis Béja
    ('Béja', 'Jendouba'): 60,
    ('Béja', 'Kef'): 65,
    ('Béja', 'Siliana'): 50,
    ('Béja', 'Sousse'): 185,
    ('Béja', 'Monastir'): 205,
    ('Béja', 'Mahdia'): 245,
    ('Béja', 'Sfax'): 315,
    ('Béja', 'Kairouan'): 155,
    ('Béja', 'Kasserine'): 245,
    ('Béja', 'Sidi Bouzid'): 260,
    ('Béja', 'Gabès'): 450,
    ('Béja', 'Médenine'): 525,
    ('Béja', 'Tataouine'): 565,
    ('Béja', 'Gafsa'): 385,
    ('Béja', 'Tozeur'): 475,
    ('Béja', 'Kébili'): 535,
    
    # Depuis Jendouba
    ('Jendouba', 'Kef'): 75,
    ('Jendouba', 'Siliana'): 85,
    ('Jendouba', 'Sousse'): 235,
    ('Jendouba', 'Monastir'): 255,
    ('Jendouba', 'Mahdia'): 295,
    ('Jendouba', 'Sfax'): 365,
    ('Jendouba', 'Kairouan'): 205,
    ('Jendouba', 'Kasserine'): 270,
    ('Jendouba', 'Sidi Bouzid'): 310,
    ('Jendouba', 'Gabès'): 500,
    ('Jendouba', 'Médenine'): 575,
    ('Jendouba', 'Tataouine'): 615,
    ('Jendouba', 'Gafsa'): 435,
    ('Jendouba', 'Tozeur'): 525,
    ('Jendouba', 'Kébili'): 585,
    
    # Depuis Kef
    ('Kef', 'Siliana'): 60,
    ('Kef', 'Sousse'): 245,
    ('Kef', 'Monastir'): 265,
    ('Kef', 'Mahdia'): 305,
    ('Kef', 'Sfax'): 375,
    ('Kef', 'Kairouan'): 170,
    ('Kef', 'Kasserine'): 155,
    ('Kef', 'Sidi Bouzid'): 280,
    ('Kef', 'Gabès'): 510,
    ('Kef', 'Médenine'): 585,
    ('Kef', 'Tataouine'): 625,
    ('Kef', 'Gafsa'): 380,
    ('Kef', 'Tozeur'): 470,
    ('Kef', 'Kébili'): 595,
    
    # Depuis Siliana
    ('Siliana', 'Sousse'): 165,
    ('Siliana', 'Monastir'): 185,
    ('Siliana', 'Mahdia'): 225,
    ('Siliana', 'Sfax'): 295,
    ('Siliana', 'Kairouan'): 95,
    ('Siliana', 'Kasserine'): 180,
    ('Siliana', 'Sidi Bouzid'): 220,
    ('Siliana', 'Gabès'): 430,
    ('Siliana', 'Médenine'): 505,
    ('Siliana', 'Tataouine'): 545,
    ('Siliana', 'Gafsa'): 315,
    ('Siliana', 'Tozeur'): 405,
    ('Siliana', 'Kébili'): 515,
    
    # Depuis Sousse
    ('Sousse', 'Monastir'): 25,
    ('Sousse', 'Mahdia'): 60,
    ('Sousse', 'Sfax'): 120,
    ('Sousse', 'Kairouan'): 50,
    ('Sousse', 'Kasserine'): 190,
    ('Sousse', 'Sidi Bouzid'): 155,
    ('Sousse', 'Gabès'): 275,
    ('Sousse', 'Médenine'): 350,
    ('Sousse', 'Tataouine'): 390,
    ('Sousse', 'Gafsa'): 230,
    ('Sousse', 'Tozeur'): 320,
    ('Sousse', 'Kébili'): 360,
    
    # Depuis Monastir
    ('Monastir', 'Mahdia'): 45,
    ('Monastir', 'Sfax'): 105,
    ('Monastir', 'Kairouan'): 70,
    ('Monastir', 'Kasserine'): 210,
    ('Monastir', 'Sidi Bouzid'): 175,
    ('Monastir', 'Gabès'): 265,
    ('Monastir', 'Médenine'): 340,
    ('Monastir', 'Tataouine'): 380,
    ('Monastir', 'Gafsa'): 250,
    ('Monastir', 'Tozeur'): 340,
    ('Monastir', 'Kébili'): 380,
    
    # Depuis Mahdia
    ('Mahdia', 'Sfax'): 90,
    ('Mahdia', 'Kairouan'): 100,
    ('Mahdia', 'Kasserine'): 240,
    ('Mahdia', 'Sidi Bouzid'): 185,
    ('Mahdia', 'Gabès'): 235,
    ('Mahdia', 'Médenine'): 310,
    ('Mahdia', 'Tataouine'): 350,
    ('Mahdia', 'Gafsa'): 260,
    ('Mahdia', 'Tozeur'): 350,
    ('Mahdia', 'Kébili'): 350,
    
    # Depuis Sfax
    ('Sfax', 'Kairouan'): 130,
    ('Sfax', 'Kasserine'): 180,
    ('Sfax', 'Sidi Bouzid'): 145,
    ('Sfax', 'Gabès'): 150,
    ('Sfax', 'Médenine'): 225,
    ('Sfax', 'Tataouine'): 280,
    ('Sfax', 'Gafsa'): 230,
    ('Sfax', 'Tozeur'): 310,
    ('Sfax', 'Kébili'): 270,
    
    # Depuis Kairouan
    ('Kairouan', 'Kasserine'): 140,
    ('Kairouan', 'Sidi Bouzid'): 105,
    ('Kairouan', 'Gabès'): 275,
    ('Kairouan', 'Médenine'): 350,
    ('Kairouan', 'Tataouine'): 390,
    ('Kairouan', 'Gafsa'): 210,
    ('Kairouan', 'Tozeur'): 300,
    ('Kairouan', 'Kébili'): 360,
    
    # Depuis Kasserine
    ('Kasserine', 'Sidi Bouzid'): 95,
    ('Kasserine', 'Gabès'): 270,
    ('Kasserine', 'Médenine'): 345,
    ('Kasserine', 'Tataouine'): 385,
    ('Kasserine', 'Gafsa'): 125,
    ('Kasserine', 'Tozeur'): 215,
    ('Kasserine', 'Kébili'): 320,
    
    # Depuis Sidi Bouzid
    ('Sidi Bouzid', 'Gabès'): 175,
    ('Sidi Bouzid', 'Médenine'): 250,
    ('Sidi Bouzid', 'Tataouine'): 290,
    ('Sidi Bouzid', 'Gafsa'): 140,
    ('Sidi Bouzid', 'Tozeur'): 230,
    ('Sidi Bouzid', 'Kébili'): 225,
    
    # Depuis Gabès
    ('Gabès', 'Médenine'): 80,
    ('Gabès', 'Tataouine'): 135,
    ('Gabès', 'Gafsa'): 165,
    ('Gabès', 'Tozeur'): 250,
    ('Gabès', 'Kébili'): 140,
    
    # Depuis Médenine
    ('Médenine', 'Tataouine'): 75,
    ('Médenine', 'Gafsa'): 240,
    ('Médenine', 'Tozeur'): 325,
    ('Médenine', 'Kébili'): 180,
    
    # Depuis Tataouine
    ('Tataouine', 'Gafsa'): 280,
    ('Tataouine', 'Tozeur'): 365,
    ('Tataouine', 'Kébili'): 200,
    
    # Depuis Gafsa
    ('Gafsa', 'Tozeur'): 90,
    ('Gafsa', 'Kébili'): 175,
    
    # Depuis Tozeur
    ('Tozeur', 'Kébili'): 95,
}

# Ajouter distances inverses
distances_complete = distances.copy()
for (ville1, ville2), dist in list(distances.items()):
    distances_complete[(ville2, ville1)] = dist

velocite_list = []

for compte in operations['compte'].unique():
    compte_ops = operations[operations['compte'] == compte].copy()
    compte_ops = compte_ops.sort_values('date_operation')
    
    velocite_max = 0
    
    for i in range(len(compte_ops) - 1):
        tx1 = compte_ops.iloc[i]
        tx2 = compte_ops.iloc[i + 1]
        
        lieu1 = tx1['Lieu']
        lieu2 = tx2['Lieu']
        
        if lieu1 == lieu2:
            continue
        
        # Temps en heures
        temps_heures = (tx2['date_operation'] - tx1['date_operation']).total_seconds() / 3600
        
        if temps_heures == 0:
            continue
        
        # Distance
        distance = distances_complete.get((lieu1, lieu2), 0)
        
        if distance > 0:
            velocite = distance / temps_heures
            velocite_max = max(velocite_max, velocite)
    
    velocite_list.append({
        'compte': compte,
        'velocite_maximale_km_h': velocite_max
    })

df_velocite = pd.DataFrame(velocite_list)
features = features.merge(df_velocite, on='compte', how='left')

print(f"✓ Vélocité maximale ajoutée")


7.  NOUVELLE VARIABLE: VÉLOCITÉ MAXIMALE (km/h)
--------------------------------------------------------------------------------
✓ Vélocité maximale ajoutée


In [9]:
print(f"✓ Vélocité maximale ajoutée")
# ============================================================================
# ÉTAPE 7  VARIABLE TPE < 100
# ============================================================================

print("\n7  VARIABLE: RATIO TPE < 100")
print("-"*80)

# Filtrer transactions TPE < 100 (Type_operation commence par T)
tpe_moins_100 = operations[
    (operations['motif operation'].str.startswith('T', na=False)) &
    (operations['montant'] < 100)
]

# Nombre de transactions TPE <100 par compte
nb_tpe_moins_100 = (
    tpe_moins_100.groupby('compte')
    .size()
    .reset_index(name='nb_tpe_moins_100')
)

# Fusion avec features
features = features.merge(nb_tpe_moins_100, on='compte', how='left')

# Remplacer NaN
features['nb_tpe_moins_100'] = features['nb_tpe_moins_100'].fillna(0)

# Calcul du ratio
features['ratio_tpe_moins_100'] = (
    features['nb_tpe_moins_100'] / features['nb_transactions']
)

print("✓ Variable ratio_tpe_moins_100 ajoutée")

✓ Vélocité maximale ajoutée

7  VARIABLE: RATIO TPE < 100
--------------------------------------------------------------------------------
✓ Variable ratio_tpe_moins_100 ajoutée


In [10]:
# ============================================================================
# VARIABLE: TRANSACTIONS PAR JOUR
# ============================================================================

# Nombre de jours actifs par compte
# Nombre de jours actifs par compte
jours_actifs = (
    operations.assign(date_only=operations['date_operation'].dt.date)  # extraire la date
    .groupby('compte')['date_only']
    .nunique()
    .reset_index(name='nb_jours_actifs')
)
# Fusion avec features
features = features.merge(jours_actifs, on='compte', how='left')

# Calcul transactions par jour
features['tx_par_jour'] = (
    features['nb_transactions'] / features['nb_jours_actifs']
)


In [12]:
# ============================================================================
# ÉTAPE 8bis: 🆕 NOUVELLE VARIABLE - RATIO CRÉDITS/DÉBITS SIMILAIRES
# ============================================================================

print("\n8bis. 🆕 NOUVELLE VARIABLE: RATIO CRÉDITS/DÉBITS SIMILAIRES")
print("-"*80)

paires_similaires_list = []

for compte in operations['compte'].unique():
    compte_ops = operations[operations['compte'] == compte].copy()
    compte_ops = compte_ops.sort_values('date_operation')

    credits = compte_ops[compte_ops['sens operation'] == 'C']['montant'].values
    debits  = compte_ops[compte_ops['sens operation'] == 'D']['montant'].values

    nb_paires = 0

    if len(credits) > 0 and len(debits) > 0:
        for c in credits:
            for d in debits:
                # Écart <= 5% entre crédit et débit → paire suspecte
                if c > 0 and abs(c - d) / c <= 0.05:
                    nb_paires += 1
                    break  # une seule paire par crédit

    total_tx = len(compte_ops)
    ratio_paires = nb_paires / total_tx if total_tx > 0 else 0

    paires_similaires_list.append({
        'compte': compte,
        'nb_paires_cd_similaires': nb_paires,
        'ratio_paires_cd_similaires': ratio_paires
    })

df_paires = pd.DataFrame(paires_similaires_list)
features = features.merge(df_paires, on='compte', how='left')
features['nb_paires_cd_similaires']    = features['nb_paires_cd_similaires'].fillna(0)
features['ratio_paires_cd_similaires'] = features['ratio_paires_cd_similaires'].fillna(0)



8bis. 🆕 NOUVELLE VARIABLE: RATIO CRÉDITS/DÉBITS SIMILAIRES
--------------------------------------------------------------------------------


In [13]:
# ============================================================================
# ÉTAPE 8: NETTOYAGE ET SAUVEGARDE
# ============================================================================

print("\n8. NETTOYAGE ET SAUVEGARDE")
print("-"*80)

# Supprimer colonnes temporaires
features = features.drop(columns=['date_min', 'date_max'], errors='ignore')

# Remplir NaN
features = features.fillna(0)

# Vérifier les comptes
print(f"\n✓ Features finales: {features.shape}")
print(f"✓ Variables créées: {len(features.columns) - 1}")
print(f"\n📋 Liste des variables:")
for col in features.columns:
    if col != 'compte':
        print(f"  - {col}")

print(f"\n✓ Exemple comptes finaux:")
print(features['compte'].head(10).tolist())

# Sauvegarder
features.to_excel(
    r"C:\Users\Pc\Desktop\features_comptes.xlsx",
    index=False
)

print(f"\n✅ FICHIER SAUVEGARDÉ: features_comptes.xlsx")
print(f"📊 Total: {len(features)} comptes, {len(features.columns)} colonnes")
print("="*80)


8. NETTOYAGE ET SAUVEGARDE
--------------------------------------------------------------------------------

✓ Features finales: (2839, 30)
✓ Variables créées: 29

📋 Liste des variables:
  - nb_transactions
  - total_montant
  - montant_moyen
  - montant_median
  - montant_std
  - montant_min
  - montant_max
  - duree_activite_jours
  - ratio_max_moyen
  - nb_transactions_mois
  - pct_nocturne
  - pct_weekend
  - nb_lieux_differents
  - lieu_principal
  - pct_lieu_principal
  - changements_lieu_frequents
  - temps_moyen_entre_tx_min
  - temps_median_entre_tx_min
  - temps_min_entre_tx_min
  - temps_std_entre_tx_min
  - montants_croissants_consecutifs
  - ratio_montant_max_min
  - velocite_maximale_km_h
  - nb_tpe_moins_100
  - ratio_tpe_moins_100
  - nb_jours_actifs
  - tx_par_jour
  - nb_paires_cd_similaires
  - ratio_paires_cd_similaires

✓ Exemple comptes finaux:
['14010123110699999232', '14010123110700003794', '14010123110700011651', '14010123110700016903', '14010123110700017388',